In [1]:
!ls ../MedRAG

figs  LICENSE  README.md  requirements.txt  src  templates


In [1]:
import json
import sys
import os
# Temporarily add MedRAG repo
sys.path.append(os.path.abspath("../MedRAG"))
from src.medrag import MedRAG
from tqdm import tqdm
import re
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

def standardize_question(text):
    pattern = r"Question: (.*?)\n(A\..*?)(\nB\..*?)(\nC\..*?)(\nD\..*)"

    match = re.search(pattern, text, re.DOTALL)
    if match:
        question = match.group(1)
        choices = {
            "A": match.group(2).strip()[3:],  # Remove 'A. ' from the beginning
            "B": match.group(3).strip()[3:],  # Remove 'B. ' from the beginning
            "C": match.group(4).strip()[3:],  # Remove 'C. ' from the beginning
            "D": match.group(5).strip()[3:]   # Remove 'D. ' from the beginning
        }
        return question, choices
    else:
        print("[ERROR]: ", text)


def get_answer(cot, q, c):
    try:
        answer, _, _ = cot.answer(question=q, options={"A": c['A'], "B": c['B'], "C": c['C'], "D": c['D']})
    except: 
        return None
    try:
        answer = json.loads(answer)
        return answer['answer_choice']
    except:
        if isinstance(answer, str):
            list_of_string = re.findall(r'"(.*?)"', answer)
            if len(list_of_string) != 0:
                return list_of_string[-1]
            print('Output is not in json format or answer is not string!')
            print(answer)
            return None
        

/home/zc347/.conda/envs/bertpy311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
cot = MedRAG(llm_name="OpenAI/gpt-4o", rationale_query=True, rag=True, retriever_name="BM25", corpus_name="PubMed_Ophthalmology+statpearls+textbooks")

/home/zc347/.conda/envs/bertpy311/lib/python3.11/site-packages/torch/cuda/__init__.py:654: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/zc347/.conda/envs/bertpy311/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/vast/palmer/home.mccleary/zc347/.conda/envs/bertpy311/lib/python3.11/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Mar 17, 2025 11:40:20 PM org.apache.lucene.store.MMapDirectory lookupProvider


In [3]:
with open('../data/results_oph/BCSC_result.json', 'r') as file:
    data = json.load(file)

In [4]:
q, c = standardize_question(data[5]['query'])
answer, snippets, snippets_score = cot.answer(question=q, options={"A": c['A'], "B": c['B'], "C": c['C'], "D": c['D']})

In [5]:
answer

'```json { "step_by_step_thinking": "To determine the treatment for recurrent epithelial erosions that may be suitable for peripheral erosions but not central erosions, we need to evaluate the suitability of each treatment option for different corneal locations. 1. Superficial keratectomy (A) involves the removal of the superficial layer of the cornea and can be used for both central and peripheral erosions. 2. Bandage contact lens (B) is used to protect the cornea and promote healing, and it can be applied to both central and peripheral areas. 3. Phototherapeutic keratectomy (C) uses an excimer laser to remove diseased tissue and is suitable for both central and peripheral erosions. 4. Anterior stromal micropuncture (D) involves creating small punctures in the anterior stroma to promote epithelial adhesion. This procedure is generally avoided in the central cornea due to the risk of inducing irregularities and scarring that can affect vision. Based on this analysis, anterior stromal m

In [11]:
data[5]

{'query': 'Given a multiple choice question in the field of ophthalmology, select the correct answer from the four options.\n\nQuestion: What treatment for recurrent epithelial erosions may be suitable for peripheral erosions but not central erosions?\nA. superficial keratectomy\nB. bandage contact lens\nC. phototherapeutic keratectomy\nD. anterior stromal micropuncture',
 'answer': 'D',
 'id': '5',
 'BCSC+OpenAI/gpt-35-turbo-16k+MedCPT+PubMed': 'Cannot be determined',
 'BCSC+OpenAI/gpt-35-turbo-16k+MedCPT+StatPearls': 'D',
 'BCSC+OpenAI/gpt-35-turbo-16k+MedCPT+Textbooks': 'B',
 'BCSC+OpenAI/gpt-35-turbo-16k+MedCPT+Wikipedia': 'Cannot be determined',
 'BCSC+OpenAI/gpt-35-turbo-16k+MedCPT+MedCorp': 'Cannot be determined',
 'BCSC+OpenAI/gpt-4o+MedCPT+PubMed': 'D',
 'BCSC+OpenAI/gpt-4o+MedCPT+StatPearls': 'D',
 'BCSC+OpenAI/gpt-4o+MedCPT+Textbooks': 'D',
 'BCSC+OpenAI/gpt-4o+MedCPT+Wikipedia': 'D',
 'BCSC+OpenAI/gpt-4o+MedCPT+MedCorp': 'D',
 'BCSC+OpenAI/gpt-35-turbo-16k+BM25+PubMed': 'D'

In [6]:
q, c = standardize_question(data[1]['query'])
answer, snippets, snippets_score = cot.answer(question=q, options={"A": c['A'], "B": c['B'], "C": c['C'], "D": c['D']})

*** { "step_by_step_thinking": "Rhegmatogenous retinal detachment (RRD) is a condition where a tear or break in the retina allows fluid to get under the retina, causing it to detach. Axial myopia is a known risk factor for RRD. To determine the next biggest risk factor after axial myopia, we need to consider the other options: A. The presence of a posterior vitreous detachment (PVD) in the eye prior to cataract surgery: PVD can lead to retinal tears, which can subsequently cause RRD. This is a significant risk factor because the vitreous pulling away from the retina can create tears. B. Paving stone degeneration: This is a peripheral retinal degeneration that is generally not associated with a high risk of RRD. C. Younger age: Younger patients are less likely to have PVD, which is more common in older individuals. Therefore, younger age is not typically a significant risk factor for RRD after cataract surgery. D. Female gender: Gender alone is not a significant risk factor for RRD. Con

In [10]:
answer

'Let\'s analyze the documents to determine the biggest risk factor for rhegmatogenous retinal detachment (RRD) after cataract surgery, aside from axial myopia. 1. **Document [0]**: This document mentions that the risk of pseudophakic retinal detachment (PPRD) after phacoemulsification cataract surgery is elevated by factors such as intraoperative vitreous loss, increasing axial length, younger age, male sex, and trainee operating surgeons. It specifically highlights younger age as a significant risk factor. 2. **Document [1]**: This document lists prominent risk factors for retinal detachment, including myopia, trauma, cataract surgery, diabetic retinopathy, and old age. However, it does not specifically address the risk factors after cataract surgery. 3. **Document [2]**: This document discusses the risk factors for retinal tear or RRD associated with acute posterior vitreous detachment (PVD). It mentions that prior cataract surgery is associated with a higher risk of RT or RRD. 4. **

In [9]:
data[1]

{'query': 'Given a multiple choice question in the field of ophthalmology, select the correct answer from the four options.\n\nQuestion: Aside from axial myopia, what is the biggest risk factor for rhegmatogenous retinal detachment (RRD) after cataract surgery?\nA. the presence of a posterior vitreous detachment in the eye prior to cataract surgery\nB. paving stone degeneration\nC. younger age\nD. female gender',
 'answer': 'C',
 'id': '1',
 'BCSC+OpenAI/gpt-35-turbo-16k+MedCPT+PubMed': 'C',
 'BCSC+OpenAI/gpt-35-turbo-16k+MedCPT+StatPearls': 'Not enough information',
 'BCSC+OpenAI/gpt-35-turbo-16k+MedCPT+Textbooks': 'A',
 'BCSC+OpenAI/gpt-35-turbo-16k+MedCPT+Wikipedia': 'C',
 'BCSC+OpenAI/gpt-35-turbo-16k+MedCPT+MedCorp': 'C',
 'BCSC+OpenAI/gpt-4o+MedCPT+PubMed': 'C',
 'BCSC+OpenAI/gpt-4o+MedCPT+StatPearls': 'A',
 'BCSC+OpenAI/gpt-4o+MedCPT+Textbooks': 'A',
 'BCSC+OpenAI/gpt-4o+MedCPT+Wikipedia': 'C',
 'BCSC+OpenAI/gpt-4o+MedCPT+MedCorp': 'C',
 'BCSC+OpenAI/gpt-35-turbo-16k+BM25+PubMed

In [8]:
snippets

[{'id': 'pubmed_ophthalmology19_797',
  'title': 'Retinal detachment following cataract phacoemulsification-a review of the literature.',
  'content': "A link between cataract surgery and rhegmatogenous retinal detachment (RRD) has long been considered. Indeed, pseudophakic retinal detachment (PPRD) forms a substantial and increasing proportion of RRD. We reviewed the literature to answer the following questions: what is the incidence of PPRD in eyes following phacoemulsification cataract surgery and how does its risk change over time following surgery? We also sought to assess how the risk is modified by intraoperative factors (operative complications, surgeon grade, subsequent laser capsulotomy), intrinsic eye-related factors (laterality, myopia, previous RRD, previous trauma, previous PVD) and patient factors (sex, age, ethnicity, affluence, systemic comorbidities). Secondarily we asked how the incidence of PPRD after phacoemulsification compares with the RRD incidence in the genera

In [26]:
data[5]

{'query': 'Given a multiple choice question in the field of ophthalmology, select the correct answer from the four options.\n\nQuestion: What treatment for recurrent epithelial erosions may be suitable for peripheral erosions but not central erosions?\nA. superficial keratectomy\nB. bandage contact lens\nC. phototherapeutic keratectomy\nD. anterior stromal micropuncture',
 'answer': 'D',
 'id': '5',
 'BCSC+OpenAI/gpt-35-turbo-16k+MedCPT+PubMed': 'Cannot be determined',
 'BCSC+OpenAI/gpt-35-turbo-16k+MedCPT+StatPearls': 'D',
 'BCSC+OpenAI/gpt-35-turbo-16k+MedCPT+Textbooks': 'B',
 'BCSC+OpenAI/gpt-35-turbo-16k+MedCPT+Wikipedia': 'Cannot be determined',
 'BCSC+OpenAI/gpt-35-turbo-16k+MedCPT+MedCorp': 'Cannot be determined',
 'BCSC+OpenAI/gpt-4o+MedCPT+PubMed': 'D',
 'BCSC+OpenAI/gpt-4o+MedCPT+StatPearls': 'D',
 'BCSC+OpenAI/gpt-4o+MedCPT+Textbooks': 'D',
 'BCSC+OpenAI/gpt-4o+MedCPT+Wikipedia': 'D',
 'BCSC+OpenAI/gpt-4o+MedCPT+MedCorp': 'D',
 'BCSC+OpenAI/gpt-35-turbo-16k+BM25+PubMed': 'D'

In [ ]:
import json
import sys
import os
# Temporarily add MedRAG repo
# sys.path.append(os.path.abspath("../MedRAG"))
from src.medrag import MedRAG
from tqdm import tqdm
import re
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

def standardize_question(text):
    pattern = r"Question: (.*?)\n(A\..*?)(\nB\..*?)(\nC\..*?)(\nD\..*)"

    match = re.search(pattern, text, re.DOTALL)
    if match:
        question = match.group(1)
        choices = {
            "A": match.group(2).strip()[3:],  # Remove 'A. ' from the beginning
            "B": match.group(3).strip()[3:],  # Remove 'B. ' from the beginning
            "C": match.group(4).strip()[3:],  # Remove 'C. ' from the beginning
            "D": match.group(5).strip()[3:]   # Remove 'D. ' from the beginning
        }
        return question, choices
    else:
        print("[ERROR]: ", text)


def get_answer(cot, q, c):
    try:
        answer, _, _ = cot.answer(question=q, options={"A": c['A'], "B": c['B'], "C": c['C'], "D": c['D']})
    except: 
        return None
    try:
        answer = json.loads(answer)
        return answer['answer_choice']
    except:
        if isinstance(answer, str):
            list_of_string = re.findall(r'"(.*?)"', answer)
            if len(list_of_string) != 0:
                return list_of_string[-1]
            print('Output is not in json format or answer is not string!')
            print(answer)
            return None
        
            
# cot_mode = True
model_names = ["OpenAI/gpt-4o"]
# model_names = ["OpenAI/gpt-4o"]
retriever_names = ["BM25", "MedCPT"]
# corpus_names = ["PubMed_Ophthalmology"]
corpus_names = ["PubMed_Ophthalmology+statpearls+textbooks"]

with open('../data/results_oph/BCSC_result.json', 'r') as file:
    data = json.load(file)


for model_name_ in model_names:
    for retriever_name_ in retriever_names:
        for corpus_name_ in corpus_names:
            output_name_ = "BCSC+" + model_name_ + "+" + retriever_name_ + "+" + corpus_name_
            print('---', output_name_, '---')
            cot = MedRAG(llm_name=model_name_, rag=True, retriever_name=retriever_name_, corpus_name=corpus_name_)
            for i, question in tqdm(enumerate(data)):
                if output_name_ in data[i].keys():
                    continue
                else:
                    q, c = standardize_question(question['query'])
                    a = get_answer(cot, q, c)
                    data[i][output_name_] = a
                    print(a)

            with open('../data/results_oph/BCSC_result.json', 'w') as file:
                json.dump(data, file)

/home/zc347/.conda/envs/bertpy311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- BCSC+OpenAI/gpt-4o+BM25+PubMed_Ophthalmology+statpearls+textbooks ---


/home/zc347/.conda/envs/bertpy311/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/vast/palmer/home.mccleary/zc347/.conda/envs/bertpy311/lib/python3.11/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Mar 06, 2025 9:37:02 PM org.apache.lucene.store.MMapDirectory lookupProvider
260it [00:00, 1001394.89it/s]


--- BCSC+OpenAI/gpt-4o+MedCPT+PubMed_Ophthalmology+statpearls+textbooks ---
No sentence-transformers model found with name /home/zc347/.cache/torch/sentence_transformers/ncbi_MedCPT-Query-Encoder. Creating a new one with CLS pooling.
No sentence-transformers model found with name /home/zc347/.cache/torch/sentence_transformers/ncbi_MedCPT-Query-Encoder. Creating a new one with CLS pooling.
No sentence-transformers model found with name /home/zc347/.cache/torch/sentence_transformers/ncbi_MedCPT-Query-Encoder. Creating a new one with CLS pooling.


1it [00:04,  4.96s/it]

...

24


30it [02:17,  4.10s/it]

22


31it [02:22,  4.46s/it]

26


In [1]:
import json
import sys
import os
# Temporarily add MedRAG repo
# sys.path.append(os.path.abspath("../MedRAG"))
from src.medrag import MedRAG
from tqdm import tqdm
import re
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

cot = MedRAG(llm_name="OpenAI/gpt-4o", rag=True, retriever_name="BM25", corpus_name="PubMed_Ophthalmology+statpearls+textbooks")

/home/zc347/.conda/envs/bertpy311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/zc347/.conda/envs/bertpy311/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/vast/palmer/home.mccleary/zc347/.conda/envs/bertpy311/lib/python3.11/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Mar 06, 2025 10:52:01 PM org.apache.lucene.store.MMapDirectory lookupProvider


In [2]:
def standardize_question(text):
    pattern = r"Question: (.*?)\n(A\..*?)(\nB\..*?)(\nC\..*?)(\nD\..*)"

    match = re.search(pattern, text, re.DOTALL)
    if match:
        question = match.group(1)
        choices = {
            "A": match.group(2).strip()[3:],  # Remove 'A. ' from the beginning
            "B": match.group(3).strip()[3:],  # Remove 'B. ' from the beginning
            "C": match.group(4).strip()[3:],  # Remove 'C. ' from the beginning
            "D": match.group(5).strip()[3:]   # Remove 'D. ' from the beginning
        }
        return question, choices
    else:
        print("[ERROR]: ", text)


def get_answer(cot, q, c):
    try:
        answer, _, _ = cot.answer(question=q, options={"A": c['A'], "B": c['B'], "C": c['C'], "D": c['D']})
    except: 
        return None
    if answer[0] != '{':
        answer = re.search(r'\{[\s\S]*\}', answer)
        answer = answer.group(0)
    try:
        answer = json.loads(answer)
        return answer['answer_choice']
    except:
        if isinstance(answer, str):
            list_of_string = re.findall(r'"(.*?)"', answer)
            if len(list_of_string) != 0:
                return list_of_string[-1]
            print('Output is not in json format or answer is not string!')
            print(answer)
            return None


with open('../data/results_oph/BCSC_result.json', 'r') as file:
    data = json.load(file)

In [3]:
q, c = standardize_question(data[0]['query'])

In [4]:
answer, _, _ = cot.answer(question=q, options={"A": c['A'], "B": c['B'], "C": c['C'], "D": c['D']})
a = get_answer(cot, q, c)

In [5]:
a

'A'

In [18]:
answer

'```json { "step_by_step_thinking": "The patient experienced a sharp pain in the left eye after hitting a nail with an electric power saw, which suggests a high-velocity injury. The presence of redness, chemosis, and a localized hemorrhage in the inferior vitreous indicates potential intraocular or intraorbital injury. Given the mechanism of injury and the symptoms, it is crucial to rule out the presence of an intraocular foreign body (IOFB) or other orbital injuries. While no foreign body is visualized on dilated examination, small or non-metallic foreign bodies might not be easily seen. Computed tomography (CT) of the orbits is the most appropriate next step as it is highly sensitive for detecting metallic foreign bodies and can provide detailed information about the extent of the injury.", "answer_choice": "A", "Rationale": ["Document [10]", "Document [22]"] } ```'

In [21]:
match = re.search(r'\{[\s\S]*\}', answer)
match = match.group(0)
json.loads(match)

{'step_by_step_thinking': 'The patient experienced a sharp pain in the left eye after hitting a nail with an electric power saw, which suggests a high-velocity injury. The presence of redness, chemosis, and a localized hemorrhage in the inferior vitreous indicates potential intraocular or intraorbital injury. Given the mechanism of injury and the symptoms, it is crucial to rule out the presence of an intraocular foreign body (IOFB) or other orbital injuries. While no foreign body is visualized on dilated examination, small or non-metallic foreign bodies might not be easily seen. Computed tomography (CT) of the orbits is the most appropriate next step as it is highly sensitive for detecting metallic foreign bodies and can provide detailed information about the extent of the injury.',
 'answer_choice': 'A',
 'Rationale': ['Document [10]', 'Document [22]']}

In [11]:
answer = json.loads(answer)
answer['answer_choice']

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
for i, question in tqdm(enumerate(data)):
    q, c = standardize_question(question['query'])
    a = get_answer(cot, q, c)
    data[i][output_name_] = a
    print(a)